In [1]:
import sys; sys.path.append("..")
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/credit_default_features.csv")

y = df["default"]
X = df.drop(columns=["default"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)
print("Train default rate:", y_train.mean().round(4))
print("Test default rate:", y_test.mean().round(4))

(23971, 40) (5993, 40)
Train default rate: 0.2213
Test default rate: 0.2213


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report

logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

logreg_pipe.fit(X_train, y_train)

y_pred = logreg_pipe.predict(X_test)
y_proba = logreg_pipe.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print(classification_report(y_test, y_pred, digits=3))

ROC-AUC: 0.754
              precision    recall  f1-score   support

           0      0.874     0.796     0.833      4667
           1      0.454     0.598     0.516      1326

    accuracy                          0.752      5993
   macro avg      0.664     0.697     0.675      5993
weighted avg      0.781     0.752     0.763      5993



In [4]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42
)

lgbm.fit(X_train, y_train)

y_pred_lgbm = lgbm.predict(X_test)
y_proba_lgbm = lgbm.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_lgbm), 4))
print(classification_report(y_test, y_pred_lgbm, digits=3))

[LightGBM] [Info] Number of positive: 5304, number of negative: 18667
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006032 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6887
[LightGBM] [Info] Number of data points in the train set: 23971, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
ROC-AUC: 0.7717
              precision    recall  f1-score   support

           0      0.875     0.811     0.842      4667
           1      0.471     0.591     0.524      1326

    accuracy                          0.763      5993
   macro avg      0.673     0.701     0.683      5993
weighted avg      0.785     0.763     0.772      5993



In [5]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "LightGBM"],
    "ROC-AUC": [
        round(roc_auc_score(y_test, y_proba), 4),
        round(roc_auc_score(y_test, y_proba_lgbm), 4)
    ]
})
print(results)

                 Model  ROC-AUC
0  Logistic Regression   0.7540
1             LightGBM   0.7717


In [6]:
import joblib
joblib.dump(logreg_pipe, "../src/logreg_model.joblib")
joblib.dump(lgbm, "../src/lgbm_model.joblib")
print("models saved")

models saved
